<a href="https://colab.research.google.com/github/gouripriya-pp/Data-Science/blob/main/XG_boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# step 1- start with a node
# step-2 - all training data pruning through teh node and prediction
# step-3- compare actual and predicted value or find residual node with tree residual values
# step 4-  sm score/quality score
#           similarity sccore=sum of residula/number of residual ^2
#           lambda-hyper parameter
# step -5-gain=leftnode+right node-rest sm score
# step-6-we pick the split with highest gain,say 3rd split
# step-7-
# step-8-final tree output=sum residula/no of residula+∝
# step-9-pruning model find residula and repeat step2.
#       final prediction-0.5+êta*sum of residula/no of residula


In [ ]:
# XG BOOST CLAASSIFIER
# --------------------
# VERY IIMPORTANT CASES.
# STEP-1- create a node that predicts a value 0.5
# STEP 2- Pass training data to this model make predictions and find residual
#         actual-predicted as per the previous
#         then create a node with it.
# STEP 3- Calculate the similarity score
#         similarity score= ∑(residula(i))^2/(pruning probabaility(i)*1-previous probabbiltiy(i))+⋋
#         previous probabbility -0.5
# STEP 4- We make all  the possibility split of the root node or pick split with highcost gain=leftnoed+right-root
#         continue process and bulid an entire tree
# STEP-5- we want to prune the tree
#         so we introduce a new parameter -cover.
#         it determines minimum number of residuals in each leaf.
#         cover=∑(previous probability *(1-pre probability))
#         there will be a fixed value for cover, by default=1,
#         if cover of a leaf<fixed even value-> we dont allow there leaf.
#         setting cover value by using hypo parameter called MIN CHILD WEIGHT.
# STEP 6- We can also  gain to prune treee
# STEP7- At last one get over final XGBOOST tree
#         leaf node-> residual
#         residual-> o/p=∑(residula(i))/∑(probability(i)(1-probabiltiy (i)+⋋))
#         log of odds-> foramt values we get

# STEP 8- build multiple trees in the sam way
# STEP 9- final o/p=0.5+rt1+rt2..............log of odd
#         log(p/1-p)=log(0.5/1-0.5)->1
#         e^final o/p/+1=e^final o/p



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb

In [9]:
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split

import xgboost as xgb

import seaborn as sns

import matplotlib.pyplot as plt






In [16]:
df=pd.read_csv('/content/Telco-Customer-Churn_XGboost.csv')
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [15]:
print(df.info())






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [13]:
columns=df.columns.tolist()

print(columns)






['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [17]:
print(df['TotalCharges'].unique())






['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']


In [20]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'])






In [19]:
df.loc[df['TotalCharges']==' ','TotalCharges']=0






In [21]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'])






In [22]:
11/11/26

0.038461538461538464

In [23]:
df.replace('','_',regex=True,inplace=True)
print(df)

      customerID  gender  SeniorCitizen Partner Dependents  tenure  \
0     7590-VHVEG  Female              0     Yes         No       1   
1     5575-GNVDE    Male              0      No         No      34   
2     3668-QPYBK    Male              0      No         No       2   
3     7795-CFOCW    Male              0      No         No      45   
4     9237-HQITU  Female              0      No         No       2   
...          ...     ...            ...     ...        ...     ...   
7038  6840-RESVB    Male              0     Yes        Yes      24   
7039  2234-XADUH  Female              0     Yes        Yes      72   
7040  4801-JZAZL  Female              0     Yes        Yes      11   
7041  8361-LTMKD    Male              1     Yes         No       4   
7042  3186-AJIEK    Male              0      No         No      66   

     PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0              No  No phone service             DSL             No  ...   
1        

In [24]:
for item in df.columns:
  print(item)
  print(df[item].unique())
  print('..........................')

customerID
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
..........................
gender
['Female' 'Male']
..........................
SeniorCitizen
[0 1]
..........................
Partner
['Yes' 'No']
..........................
Dependents
['No' 'Yes']
..........................
tenure
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
..........................
PhoneService
['No' 'Yes']
..........................
MultipleLines
['No phone service' 'No' 'Yes']
..........................
InternetService
['DSL' 'Fiber optic' 'No']
..........................
OnlineSecurity
['No' 'Yes' 'No internet service']
..........................
OnlineBackup
['Yes' 'No' 'No internet service']
..........................
DeviceProtection
['No' 'Yes' 'No internet service']
.....................

In [ ]:
pd._dummies=pd.get_dummies(df.dtype='int 32',drop_first=True)

In [ ]:
columns=df.columns.tolist()
print(columns)

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [25]:
ml_dummies = pd.get_dummies(
    df[['gender', 'Partner', 'Dependents', 'PhoneService',
        'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod', 'Churn']],
    dtype='int32',
    drop_first=True
)


In [26]:
df.drop(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
        'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod', 'Churn'],axis=1,inplace=True)  # dropping encoded columns

In [27]:
final_data=pd.concat([df,ml_dummies],axis=1)

In [28]:
print(final_data)

      SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0                 0       1           29.85         29.85            0   
1                 0      34           56.95       1889.50            1   
2                 0       2           53.85        108.15            1   
3                 0      45           42.30       1840.75            1   
4                 0       2           70.70        151.65            0   
...             ...     ...             ...           ...          ...   
7038              0      24           84.80       1990.50            1   
7039              0      72          103.20       7362.90            0   
7040              0      11           29.60        346.45            0   
7041              1       4           74.40        306.60            1   
7042              0      66          105.65       6844.50            1   

      Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0               1               0                 0   
1

In [29]:
print(final_data)

      SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0                 0       1           29.85         29.85            0   
1                 0      34           56.95       1889.50            1   
2                 0       2           53.85        108.15            1   
3                 0      45           42.30       1840.75            1   
4                 0       2           70.70        151.65            0   
...             ...     ...             ...           ...          ...   
7038              0      24           84.80       1990.50            1   
7039              0      72          103.20       7362.90            0   
7040              0      11           29.60        346.45            0   
7041              1       4           74.40        306.60            1   
7042              0      66          105.65       6844.50            1   

      Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0               1               0                 0   
1

In [30]:
print(final_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   int64  
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   gender_Male                            7043 non-null   int32  
 5   Partner_Yes                            7043 non-null   int32  
 6   Dependents_Yes                         7043 non-null   int32  
 7   PhoneService_Yes                       7043 non-null   int32  
 8   MultipleLines_No phone service         7043 non-null   int32  
 9   MultipleLines_Yes                      7043 non-null   int32  
 10  InternetService_Fiber optic            7043 non-null   int32  
 11  Inte

In [31]:
final_data['Churn_Yes'].value_counts()

,count
Churn_Yes,
0,5174
1,1869


In [32]:
x = final_data.drop('Churn_Yes', axis=1)
y = final_data['Churn_Yes']

In [33]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [34]:
import xgboost as xgb
model=xgb.XGBClassifier()

model.fit(xtrain, ytrain)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [35]:
ypred = model.predict(xtest)

print(ypred)

[0 1 0 ... 0 0 0]


In [36]:
print('Training score:', model.score(xtrain, ytrain))

print('Testing score:', model.score(xtest, ytest)) # over fitt so avoid it with fine tume

Training score: 0.9305999290024849
Testing score: 0.7764371894960965


In [37]:
model=xgb.XGBClassifier(subsample=0.9,colsample_bytree=0.5)
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth':[3,5],'leaerning_rate':[0.01,0.05,0.07,0.09],'gamma':[0.0,0.1,0.25],'reg_lambda':[0.0,0.1,0.25],'scale_pos_weight':[1,2,3]}
gridsearch=GridSearchCV(estimator=model,param_grid=param_grid,cv=3,n_jobs=-1,verbose=1,scoring='roc_auc')
gridsearch.fit(xtrain,ytrain)

Fitting 3 folds for each of 216 candidates, totalling 648 fits


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [12:45:23] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "leaerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=0.5, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'gamma': [0.0, 0.1, 0.25],
                         'leaerning_rate': [0.01, 0.05, 0.07, 0.09],
                         'max_depth': [3, 5], 'reg_lambda': [0.0, 0.1, 0.25],
                         'scale_pos_weight': [1, 2, 3]},
             scoring='roc_auc', verbose=1)

In [38]:
print(gridsearch.best_params_)

{'gamma': 0.25, 'leaerning_rate': 0.01, 'max_depth': 3, 'reg_lambda': 0.0, 'scale_pos_weight': 3}


In [39]:
best_model=gridsearch.best_estimator_
print(f'training score:{best_model.score(xtrain,ytrain)}')
print(f'testing score:{best_model.score(xtest,ytest)}')

training score:0.7896698615548455
testing score:0.7423704755145494
